# 03 — Weighted Average FC

Compute each subject's occupancy-weighted average FC using mean HMM state occupancies (posterior gamma), without behavioral data. Aligned with `graph_theory` notebook `04_weighted_average_fc.ipynb`.


In [1]:
import os
from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

with open(PROJECT_ROOT / "config.yaml", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

    DATA_MODE = CONFIG.get("data_mode", "example")
    N_STATES = CONFIG["n_states"]
    N_TIMEPOINTS = CONFIG["n_timepoints"]
    N_SUBJECTS = CONFIG["n_subjects"]
    METHOD = CONFIG["method"]
    USE_GAMMA = CONFIG["use_gamma"]
    MIN_EFF_N = CONFIG["min_eff_n"]
    MIN_HARD_N = CONFIG["min_hard_n"]
    GRAD = CONFIG["gradient"]

    DATA_ROOT = PROJECT_ROOT / (CONFIG["paths"]["example_root"] if DATA_MODE == "example" else CONFIG["paths"]["study_data_root"])
    OUTPUT_ROOT = PROJECT_ROOT / CONFIG["paths"]["output_root"] / DATA_MODE
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    print(f"project={PROJECT_ROOT} mode={DATA_MODE} out={OUTPUT_ROOT}")

    import json
    import numpy as np
    import pandas as pd

    with open(OUTPUT_ROOT / "analysis_metadata.json", encoding="utf-8") as f:
        K = int(json.load(f)["K"])

        fc_npz = np.load(OUTPUT_ROOT / "02_state_fc" / "per_subject_state_FC.npz", allow_pickle=True)
        per_subject_state_FC = fc_npz["per_subject_state_FC"].item()
        summary_df = pd.read_csv(OUTPUT_ROOT / "02_state_fc" / "per_subject_state_FC_summary.csv")

        print(f"Subjects with state FC: {len(per_subject_state_FC)}")

project=d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis mode=example out=d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis\results\example


Subjects with state FC: 5


In [2]:
def compute_state_weights_from_summary(summary_df, subj, K):
    subj_rows = summary_df[summary_df["subj"].astype(str) == str(subj)]
    total = subj_rows["eff_n_or_n"].sum()
    weights = {state: 0.0 for state in range(1, K + 1)}
    if total <= 0:
        return weights
        for _, row in subj_rows.iterrows():
            weights[int(row["state"])] = float(row["eff_n_or_n"]) / float(total)
            return weights


    def calculate_weighted_average_fc(per_subject_state_FC, summary_df, K):
        weighted_avg_FC = {}
        for subj, subj_states in per_subject_state_FC.items():
            weights = compute_state_weights_from_summary(summary_df, subj, K)
            weighted_avg = None
            total_weight = 0.0
            for state in range(1, K + 1):
                state_idx = state - 1
                weight = weights[state]
                if state_idx not in subj_states or subj_states[state_idx] is None or weight <= 0:
                    continue
                FC = subj_states[state_idx]
                if np.all(np.isnan(FC)):
                    continue
                FC_clean = np.nan_to_num(FC, nan=0.0)
                if weighted_avg is None:
                    weighted_avg = np.zeros_like(FC_clean)
                    weighted_avg += FC_clean * weight
                    total_weight += weight
                    weighted_avg_FC[subj] = (weighted_avg / total_weight) if total_weight > 0 else None
                return weighted_avg_FC


                weighted_avg_FC = calculate_weighted_average_fc(per_subject_state_FC, summary_df, K)
                n_valid = sum(1 for fc in weighted_avg_FC.values() if fc is not None)
                print(f"Weighted average FC ready for {n_valid} subjects.")

                out_dir = OUTPUT_ROOT / "03_weighted_fc"
                out_dir.mkdir(parents=True, exist_ok=True)
                np.savez_compressed(
                out_dir / "weighted_average_FC.npz",
                allow_pickle=True,
                weighted_avg_FC=weighted_avg_FC,
                )
                print(f"Saved weighted_average_FC.npz -> {out_dir}")

Weighted average FC ready for 5 subjects.
Saved weighted_average_FC.npz -> d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis\results\example\03_weighted_fc
